# Notebook 8 — SQL Analytics on Delta Lake
**Project**: Real-Time Retail Analytics & Demand Prediction Platform

**Author**: Vineet Joshi | ZDA25M007 | IIT Madras Zanzibar

**Stack**: Spark SQL | DuckDB | Delta Lake | MinIO

Runs SQL queries on Delta Lake tables for business analytics. Demonstrates both Spark SQL (distributed) and DuckDB (fast local analytics).

## 8.1 Install and Config

In [1]:
import sys
!{sys.executable} -m pip install duckdb deltalake pyarrow -q
print('Done.')

Done.


In [2]:
import pandas as pd
import numpy as np
import duckdb
import warnings
warnings.filterwarnings('ignore')
from deltalake import DeltaTable

STORAGE = {
    'AWS_ENDPOINT_URL':           'http://minio:9000',
    'AWS_ACCESS_KEY_ID':          'admin',
    'AWS_SECRET_ACCESS_KEY':      'bigdata123',
    'AWS_ALLOW_HTTP':             'true',
    'AWS_S3_ALLOW_UNSAFE_RENAME': 'true',
    'AWS_REGION':                 'us-east-1'
}

print('Config loaded.')

Config loaded.


## 8.2 Load Delta Lake Tables

In [3]:
transactions = DeltaTable('s3://retail-v2/delta/transactions', storage_options=STORAGE).to_pandas()
features     = DeltaTable('s3://retail-v2/delta/features',     storage_options=STORAGE).to_pandas()
predictions  = DeltaTable('s3://retail-v2/delta/predictions',  storage_options=STORAGE).to_pandas()

print(f'Tables loaded from Delta Lake (retail-v2 bucket):')
print(f'  transactions : {len(transactions):,} rows')
print(f'  features     : {len(features):,} rows')
print(f'  predictions  : {len(predictions):,} rows')

Tables loaded from Delta Lake (retail-v2 bucket):
  transactions : 500,000 rows
  features     : 272,535 rows
  predictions  : 54,507 rows


## 8.3 Register Tables in DuckDB

In [4]:
con = duckdb.connect()

con.register('transactions', transactions)
con.register('features', features)
con.register('predictions', predictions)

print(con.sql('SHOW TABLES').df())
print('\nAll tables registered in DuckDB.')

           name
0      features
1   predictions
2  transactions

All tables registered in DuckDB.


## 8.4 SQL Query 1: Total Revenue by Country (Top 10)

In [5]:
q1 = con.sql("""
    SELECT 
        Country,
        COUNT(*) AS TotalTransactions,
        ROUND(SUM(Revenue), 2) AS TotalRevenue,
        ROUND(AVG(Revenue), 2) AS AvgRevenue,
        COUNT(DISTINCT CustomerID) AS UniqueCustomers
    FROM transactions
    GROUP BY Country
    ORDER BY TotalRevenue DESC
    LIMIT 10
""")

print('Query 1: Top 10 Countries by Revenue')
print('='*70)
q1.df()

Query 1: Top 10 Countries by Revenue


,Country,TotalTransactions,TotalRevenue,AvgRevenue,UniqueCustomers
0,United Kingdom,454795,9102268.72,20.01,4264
1,EIRE,10301,425366.76,41.29,5
2,Netherlands,3250,344278.27,105.93,22
3,Germany,9645,250936.12,26.02,77
4,France,6932,178321.82,25.72,58
5,Denmark,498,80553.71,161.75,10
6,Sweden,1050,67291.62,64.09,16
7,Spain,1655,56183.34,33.95,30
8,Switzerland,1339,48169.25,35.97,15
9,Australia,780,45621.14,58.49,15


## 8.5 SQL Query 2: Monthly Revenue Trend

In [6]:
q2 = con.sql("""
    SELECT 
        Year,
        Month,
        ROUND(SUM(Revenue), 2) AS MonthlyRevenue,
        SUM(CAST(Quantity AS BIGINT)) AS MonthlyQuantity,
        COUNT(DISTINCT InvoiceNo) AS Invoices
    FROM transactions
    GROUP BY Year, Month
    ORDER BY Year, Month
""")

print('Query 2: Monthly Revenue Trend')
print('='*60)
q2.df()

Query 2: Monthly Revenue Trend


,Year,Month,MonthlyRevenue,MonthlyQuantity,Invoices
0,2009,12,686465.75,400153.0,1512
1,2010,1,675937.64,433958.0,1009
2,2010,2,1012030.37,746576.0,1148
3,2010,3,1395049.51,1003730.0,1575
4,2010,4,645787.78,382676.0,1390
5,2010,5,610932.36,393192.0,1413
6,2010,6,621579.33,380931.0,1493
7,2010,7,609326.22,336398.0,1450
8,2010,8,590780.57,446946.0,1288
9,2010,9,824107.92,562476.0,1735


## 8.6 SQL Query 3: Top 15 Best-Selling Products

In [7]:
q3 = con.sql("""
    SELECT 
        StockCode,
        Description,
        SUM(CAST(Quantity AS BIGINT)) AS TotalQtySold,
        ROUND(SUM(Revenue), 2) AS TotalRevenue,
        COUNT(DISTINCT CustomerID) AS BuyerCount
    FROM transactions
    GROUP BY StockCode, Description
    ORDER BY TotalQtySold DESC
    LIMIT 15
""")

print('Query 3: Top 15 Best-Selling Products')
print('='*70)
q3.df()

Query 3: Top 15 Best-Selling Products


,StockCode,Description,TotalQtySold,TotalRevenue,BuyerCount
0,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215.0,79995.36,1
1,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,67381.0,13861.33,284
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,64885.0,173391.00,1124
3,21212,PACK OF 72 RETRO SPOT CAKE CASES,55043.0,26582.02,632
4,17003,BROCADE RING PURSE,50241.0,9183.06,125
5,84879,ASSORTED COLOUR BIRD ORNAMENT,48564.0,76972.51,607
6,37410,BLACK AND WHITE PAISLEY FLOWER MUG,44951.0,5089.32,13
7,84991,60 TEATIME FAIRY CAKE CASES,43028.0,20910.55,588
8,21977,PACK OF 60 PINK PAISLEY CAKE CASES,42115.0,20579.80,573
9,21982,PACK OF 12 SUKI TISSUES,36875.0,11315.40,210


## 8.7 SQL Query 4: Hourly Sales Pattern

In [8]:
q4 = con.sql("""
    SELECT 
        Hour,
        COUNT(*) AS Transactions,
        ROUND(SUM(Revenue), 2) AS Revenue,
        ROUND(AVG(CAST(Quantity AS DOUBLE)), 2) AS AvgQuantity
    FROM transactions
    GROUP BY Hour
    ORDER BY Hour
""")

print('Query 4: Hourly Sales Pattern')
print('='*50)
q4.df()

Query 4: Hourly Sales Pattern


,Hour,Transactions,Revenue,AvgQuantity
0,0,20883,450703.35,13.50
1,1,20926,450158.19,12.42
2,2,20682,449717.23,12.75
3,3,20786,463190.52,12.75
4,4,20954,464280.12,14.79
5,5,20903,438828.85,12.99
6,6,20931,458244.70,13.11
7,7,20895,453225.22,13.39
8,8,20946,438803.44,14.50
9,9,20730,450685.69,14.35


## 8.8 SQL Query 5: Customer Segmentation (RFM)

In [9]:
q5 = con.sql("""
    SELECT 
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS Frequency,
        ROUND(SUM(Revenue), 2) AS MonetaryValue,
        MAX(Date) AS LastPurchase,
        MIN(Date) AS FirstPurchase,
        ROUND(AVG(Revenue), 2) AS AvgOrderValue
    FROM transactions
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID
    ORDER BY MonetaryValue DESC
    LIMIT 15
""")

print('Query 5: Top 15 Customers by Monetary Value (RFM)')
print('='*75)
q5.df()

Query 5: Top 15 Customers by Monetary Value (RFM)


,CustomerID,Frequency,MonetaryValue,LastPurchase,FirstPurchase,AvgOrderValue
0,18102,90,408704.06,2011-02-07,2009-12-01,539.90
1,14646,88,322201.87,2011-02-23,2009-12-02,144.10
2,14156,114,226777.05,2011-03-16,2009-12-01,73.53
3,14911,228,188794.43,2011-03-21,2009-12-01,27.64
4,13694,105,177716.25,2011-03-24,2009-12-04,133.22
5,15061,96,101802.83,2011-03-24,2009-12-01,146.48
6,17511,36,99859.71,2011-03-17,2009-12-02,87.98
7,16684,33,90597.16,2011-03-19,2009-12-07,186.41
8,17949,74,82473.44,2010-12-03,2009-12-02,710.98
9,12346,12,80422.92,2011-01-18,2009-12-14,2010.57


## 8.9 SQL Query 6: Day-of-Week Revenue

In [10]:
q6 = con.sql("""
    SELECT 
        DayName AS DayOfWeek,
        COUNT(*) AS Transactions,
        ROUND(SUM(Revenue), 2) AS TotalRevenue,
        ROUND(AVG(Revenue), 2) AS AvgRevenue,
        SUM(CAST(Quantity AS BIGINT)) AS TotalQuantity
    FROM transactions
    GROUP BY DayName
    ORDER BY TotalRevenue DESC
""")

print('Query 6: Revenue by Day of Week')
print('='*65)
q6.df()

Query 6: Revenue by Day of Week


,DayOfWeek,Transactions,TotalRevenue,AvgRevenue,TotalQuantity
0,Wednesday,85529,2016382.68,23.58,1280298.0
1,Friday,86218,2012420.46,23.34,1191726.0
2,Thursday,87544,2001119.21,22.86,1275139.0
3,Tuesday,81001,1955461.10,24.14,1385801.0
4,Monday,85310,1520681.39,17.83,1021873.0
5,Saturday,32804,796265.18,24.27,458861.0
6,Sunday,41594,579244.75,13.93,325144.0


## 8.10 SQL Query 7: Prediction Accuracy by Country

In [11]:
q7 = con.sql("""
    SELECT 
        Country,
        COUNT(*) AS NumPredictions,
        ROUND(AVG(TotalQuantity), 2) AS AvgActual,
        ROUND(AVG(LR_Prediction), 2) AS AvgLR,
        ROUND(AVG(RF_Prediction), 2) AS AvgRF,
        ROUND(AVG(GBT_Prediction), 2) AS AvgGBT,
        ROUND(AVG(ABS(TotalQuantity - LR_Prediction)), 2) AS LR_MAE,
        ROUND(AVG(ABS(TotalQuantity - RF_Prediction)), 2) AS RF_MAE,
        ROUND(AVG(ABS(TotalQuantity - GBT_Prediction)), 2) AS GBT_MAE
    FROM predictions
    GROUP BY Country
    ORDER BY NumPredictions DESC
    LIMIT 10
""")

print('Query 7: Prediction Accuracy by Country')
print('='*90)
q7.df()

Query 7: Prediction Accuracy by Country


,Country,NumPredictions,AvgActual,AvgLR,AvgRF,AvgGBT,LR_MAE,RF_MAE,GBT_MAE
0,United Kingdom,46117,22.02,21.03,20.42,20.35,18.29,4.75,2.71
1,France,1627,13.46,28.13,13.76,13.43,16.42,1.92,0.43
2,Germany,1560,16.44,33.94,16.60,16.43,20.15,2.20,0.54
3,EIRE,1368,16.41,33.22,17.00,16.49,25.02,2.02,0.57
4,Netherlands,579,62.61,83.92,63.25,62.85,45.04,6.05,1.88
5,Belgium,370,15.09,29.57,15.25,15.06,18.65,1.98,0.48
6,Spain,318,17.63,30.79,18.51,17.81,15.73,2.25,0.56
7,Switzerland,307,20.61,43.36,21.65,20.58,26.50,2.26,0.61
8,Norway,287,26.33,31.93,27.22,26.15,17.96,2.15,0.70
9,Portugal,272,13.80,28.60,14.33,13.80,18.17,2.17,0.46


## 8.11 SQL Query 8: Pareto Analysis (CTE + Window)

In [12]:
q8 = con.sql("""
    WITH product_revenue AS (
        SELECT 
            StockCode,
            Description,
            SUM(Revenue) AS ProductRevenue
        FROM transactions
        GROUP BY StockCode, Description
    ),
    ranked AS (
        SELECT *,
            SUM(ProductRevenue) OVER (ORDER BY ProductRevenue DESC) AS CumulativeRevenue,
            SUM(ProductRevenue) OVER () AS GrandTotal,
            ROW_NUMBER() OVER (ORDER BY ProductRevenue DESC) AS Rank
        FROM product_revenue
    )
    SELECT 
        Rank,
        StockCode,
        Description,
        ROUND(ProductRevenue, 2) AS Revenue,
        ROUND(CumulativeRevenue / GrandTotal * 100, 2) AS CumulativePct
    FROM ranked
    WHERE Rank <= 20
    ORDER BY Rank
""")

print('Query 8: Pareto Analysis - Top 20 Products')
print('='*75)
q8.df()

Query 8: Pareto Analysis - Top 20 Products


,Rank,StockCode,Description,Revenue,CumulativePct
0,1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,173391.00,1.59
1,2,22423,REGENCY CAKESTAND 3 TIER,167543.04,3.13
2,3,M,Manual,144415.86,4.46
3,4,23166,MEDIUM CERAMIC TOP STORAGE JAR,79995.36,5.20
4,5,84879,ASSORTED COLOUR BIRD ORNAMENT,76972.51,5.90
5,6,POST,POSTAGE,58326.50,6.44
6,7,85099B,JUMBO BAG RED RETROSPOT,54899.97,6.94
7,8,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,43122.01,7.34
8,9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,41479.21,7.72
9,10,47566,PARTY BUNTING,40968.99,8.10


## 8.12 SQL Query 9: Seasonal Demand Patterns

In [13]:
q9 = con.sql("""
    SELECT 
        CASE 
            WHEN Month IN (12, 1, 2)  THEN 'Winter'
            WHEN Month IN (3, 4, 5)   THEN 'Spring'
            WHEN Month IN (6, 7, 8)   THEN 'Summer'
            WHEN Month IN (9, 10, 11) THEN 'Autumn'
        END AS Season,
        COUNT(*) AS Transactions,
        ROUND(SUM(Revenue), 2) AS TotalRevenue,
        SUM(CAST(Quantity AS BIGINT)) AS TotalQuantity,
        ROUND(AVG(UnitPrice), 2) AS AvgPrice,
        COUNT(DISTINCT StockCode) AS UniqueProducts
    FROM transactions
    GROUP BY Season
    ORDER BY TotalRevenue DESC
""")

print('Query 9: Seasonal Demand Patterns')
print('='*70)
q9.df()

Query 9: Seasonal Demand Patterns


,Season,Transactions,TotalRevenue,TotalQuantity,AvgPrice,UniqueProducts
0,Winter,142800,3268633.53,2108284.0,3.34,3608
1,Autumn,143401,3016436.01,1811797.0,3.20,3135
2,Spring,129808,2774819.10,1854486.0,3.57,3214
3,Summer,83991,1821686.12,1164275.0,3.22,3016


## 8.13 SQL Query 10: Cross-Country Product Ranking (Window Function)

In [14]:
q10 = con.sql("""
    WITH country_product AS (
        SELECT 
            Country,
            StockCode,
            Description,
            SUM(CAST(Quantity AS BIGINT)) AS TotalQty,
            RANK() OVER (PARTITION BY Country ORDER BY SUM(CAST(Quantity AS BIGINT)) DESC) AS ProductRank
        FROM transactions
        GROUP BY Country, StockCode, Description
    )
    SELECT Country, StockCode, Description, TotalQty, ProductRank
    FROM country_product
    WHERE ProductRank <= 3
    ORDER BY Country, ProductRank
    LIMIT 30
""")

print('Query 10: Top 3 Products per Country (Window Function)')
print('='*75)
q10.df()

Query 10: Top 3 Products per Country (Window Function)


,Country,StockCode,Description,TotalQty,ProductRank
0,Australia,22630,DOLLY GIRL LUNCH BOX,804.0,1
1,Australia,22492,MINI PAINT SET VINTAGE,684.0,2
2,Australia,21086,SET/6 RED SPOTTY PAPER CUPS,648.0,3
3,Austria,22546,MINI JIGSAW PURDEY,240.0,1
4,Austria,84879,ASSORTED COLOUR BIRD ORNAMENT,200.0,2
5,Austria,16033,MINI HIGHLIGHTER PENS,120.0,3
6,Bahrain,22422,TOOTHPASTE TUBE PEN,60.0,1
7,Bahrain,22421,LIPSTICK PEN FUSCHIA,24.0,2
8,Bahrain,21789,KIDS RAIN MAC PINK,24.0,2
9,Bahrain,20871,OPULENT VELVET SET/3 CANDLES,24.0,2


## 8.14 Spark SQL Section

In [15]:
from pyspark.sql import SparkSession

PACKAGES = (
    'io.delta:delta-spark_2.12:3.2.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4,'
    'com.amazonaws:aws-java-sdk-bundle:1.12.367'
)

spark = (
    SparkSession.builder
    .appName('RetailSQL')
    .master('local[*]')
    .config('spark.jars.packages', PACKAGES)
    .config('spark.eventLog.enabled', 'false')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', 'admin')
    .config('spark.hadoop.fs.s3a.secret.key', 'bigdata123')
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark SQL ready ({spark.version})')

Spark SQL ready (3.5.0)


In [16]:
spark.read.format('delta').load('s3a://retail-v2/delta/spark_features').createOrReplaceTempView('spark_features')
spark.read.format('delta').load('s3a://retail-v2/delta/spark_country_agg').createOrReplaceTempView('country_agg')
spark.read.format('delta').load('s3a://retail-v2/delta/spark_monthly').createOrReplaceTempView('monthly')
spark.read.format('delta').load('s3a://retail-v2/delta/spark_daily_ma').createOrReplaceTempView('daily_ma')

print('Spark SQL views registered:')
spark.sql('SHOW TABLES').show()

Spark SQL views registered:
+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |   country_agg|      false|
|         |      daily_ma|      false|
|         |       monthly|      false|
|         |spark_features|      false|
+---------+--------------+-----------+



### Spark SQL Query 1: Feature Statistics by Country

In [17]:
print('Spark SQL: Feature Statistics by Country')
spark.sql("""
    SELECT 
        Country,
        COUNT(*) AS FeatureRows,
        ROUND(AVG(DailyQty), 2) AS AvgDailyQty,
        ROUND(AVG(DailyRevenue), 2) AS AvgDailyRevenue,
        ROUND(AVG(Avg7DayQty), 2) AS Avg7DayQty
    FROM spark_features
    GROUP BY Country
    ORDER BY FeatureRows DESC
    LIMIT 10
""").show(truncate=False)

Spark SQL: Feature Statistics by Country
+--------------+-----------+-----------+---------------+----------+
|Country       |FeatureRows|AvgDailyQty|AvgDailyRevenue|Avg7DayQty|
+--------------+-----------+-----------+---------------+----------+
|United Kingdom|429564     |20.27      |34.29          |20.6      |
|Germany       |15749      |14.48      |27.39          |14.63     |
|EIRE          |15230      |21.12      |40.84          |22.57     |
|France        |13036      |20.99      |27.25          |20.23     |
|Netherlands   |5017       |76.54      |110.37         |73.4      |
|Spain         |3605       |14.09      |30.36          |14.73     |
|Belgium       |3037       |11.62      |21.69          |11.57     |
|Switzerland   |3001       |17.44      |33.43          |17.53     |
|Portugal      |2359       |11.92      |24.28          |12.45     |
|Australia     |1789       |58.18      |94.82          |52.76     |
+--------------+-----------+-----------+---------------+----------+



### Spark SQL Query 2: Daily Revenue with Moving Average

In [18]:
print('Spark SQL: Daily Revenue with 7-Day Moving Average')
spark.sql("""
    SELECT 
        Date,
        ROUND(DailyRevenue, 2) AS Revenue,
        ROUND(MA_7day, 2) AS MovingAvg7Day,
        ROUND(CumulativeRevenue, 2) AS CumulativeTotal
    FROM daily_ma
    ORDER BY Date DESC
    LIMIT 15
""").show(truncate=False)

Spark SQL: Daily Revenue with 7-Day Moving Average
+----------+---------+-------------+---------------+
|Date      |Revenue  |MovingAvg7Day|CumulativeTotal|
+----------+---------+-------------+---------------+
|2011-12-10|180063.18|61835.0      |1.774992645E7  |
|2011-12-09|40485.24 |39424.2      |1.756986328E7  |
|2011-12-08|53319.88 |40125.01     |1.752937803E7  |
|2011-12-07|58316.61 |38536.24     |1.747605815E7  |
|2011-12-06|58466.61 |37001.12     |1.741774154E7  |
|2011-12-05|32754.53 |35977.29     |1.735927493E7  |
|2011-12-04|9438.95  |35818.03     |1.73265204E7   |
|2011-12-03|23187.54 |35549.6      |1.731708145E7  |
|2011-12-02|45390.93 |34313.31     |1.72938939E7   |
|2011-12-01|42198.47 |32464.41     |1.724850297E7  |
|2011-11-30|47570.77 |34308.47     |1.72063045E7   |
|2011-11-29|51299.85 |36401.34     |1.715873373E7  |
|2011-11-28|31639.66 |36146.45     |1.710743388E7  |
|2011-11-27|7559.94  |36454.94     |1.707579422E7  |
|2011-11-26|14533.54 |37496.63     |1.706823428E

### Spark SQL Query 3: Monthly Growth Rate

In [19]:
print('Spark SQL: Monthly Revenue with Growth Rate')
spark.sql("""
    SELECT 
        Year, Month,
        ROUND(MonthlyRevenue, 2) AS Revenue,
        MonthlyQty AS Quantity,
        MonthlyInvoices AS Invoices,
        ROUND(
            (MonthlyRevenue - LAG(MonthlyRevenue) OVER (ORDER BY Year, Month)) 
            / LAG(MonthlyRevenue) OVER (ORDER BY Year, Month) * 100
        , 2) AS GrowthPct
    FROM monthly
    ORDER BY Year, Month
""").show(30, truncate=False)

Spark SQL: Monthly Revenue with Growth Rate
+----+-----+----------+--------+--------+---------+
|Year|Month|Revenue   |Quantity|Invoices|GrowthPct|
+----+-----+----------+--------+--------+---------+
|2009|12   |686465.75 |400153  |1512    |NULL     |
|2010|1    |548184.45 |365774  |1009    |-20.14   |
|2010|2    |506015.19 |373288  |1148    |-7.69    |
|2010|3    |697524.75 |501865  |1575    |37.85    |
|2010|4    |595581.91 |351351  |1390    |-14.61   |
|2010|5    |610932.36 |393192  |1413    |2.58     |
|2010|6    |621579.33 |380931  |1493    |1.74     |
|2010|7    |609326.22 |336398  |1450    |-1.97    |
|2010|8    |590780.57 |446946  |1288    |-3.04    |
|2010|9    |824107.92 |562476  |1735    |39.49    |
|2010|10   |1043095.67|602758  |2202    |26.57    |
|2010|11   |1161563.1 |652036  |2657    |11.36    |
|2010|12   |910410.35 |483451  |1510    |-21.62   |
|2011|1    |563253.29 |342894  |983     |-38.13   |
|2011|2    |448868.85 |267485  |1045    |-20.31   |
|2011|3    |589819.8

## 8.15 Summary

In [20]:
spark.stop()
con.close()

print('='*55)
print('NOTEBOOK 8 COMPLETE')
print('='*55)
print('  DuckDB queries   : 10 business analytics queries')
print('  Spark SQL queries: 3 distributed queries')
print('  Tables queried   : transactions, features, predictions')
print('  SQL features     : GROUP BY, Window Functions,')
print('                     CTEs, CASE WHEN, LAG, RANK, Pareto')
print()
print('Query types covered:')
print('  1.  Country revenue analysis')
print('  2.  Monthly revenue trend')
print('  3.  Best-selling products')
print('  4.  Hourly sales pattern')
print('  5.  Customer segmentation (RFM)')
print('  6.  Day-of-week analysis')
print('  7.  Prediction accuracy by country')
print('  8.  Pareto analysis (revenue concentration)')
print('  9.  Seasonal demand patterns')
print('  10. Cross-country product ranking (window func)')
print('  11. Spark SQL feature statistics')
print('  12. Spark SQL daily revenue + moving avg')
print('  13. Spark SQL monthly growth rate')
print()
print('Next: Notebook 9 - Real-Time Interactive Dashboard')

NOTEBOOK 8 COMPLETE
  DuckDB queries   : 10 business analytics queries
  Spark SQL queries: 3 distributed queries
  Tables queried   : transactions, features, predictions
  SQL features     : GROUP BY, Window Functions,
                     CTEs, CASE WHEN, LAG, RANK, Pareto

Query types covered:
  1.  Country revenue analysis
  2.  Monthly revenue trend
  3.  Best-selling products
  4.  Hourly sales pattern
  5.  Customer segmentation (RFM)
  6.  Day-of-week analysis
  7.  Prediction accuracy by country
  8.  Pareto analysis (revenue concentration)
  9.  Seasonal demand patterns
  10. Cross-country product ranking (window func)
  11. Spark SQL feature statistics
  12. Spark SQL daily revenue + moving avg
  13. Spark SQL monthly growth rate

Next: Notebook 9 - Real-Time Interactive Dashboard
